In [ ]:
from openmm.app import *
from openmm import *
from openmm.unit import *
from sys import stdout
from openmmtools.integrators import LangevinIntegrator, LangevinSplittingGirsanov
from openmmplumed import PlumedForce
import matplotlib

from deeptime.decomposition import TICA

import numpy as np
import MDAnalysis as md
from matplotlib import pyplot as plt

from deeptime.clustering import MiniBatchKMeans
from deeptime.markov import TransitionCountEstimator
from deeptime.markov.msm import MaximumLikelihoodMSM
from deeptime.plots import plot_implied_timescales
from deeptime.util.validation import implied_timescales
from deeptime.markov import GirsanovReweightingEstimator

from scipy.linalg import eig
from scipy.stats import gaussian_kde

from pymbar import MBAR
import plumed
import subprocess

from scipy.interpolate import CubicSpline

import gc

import warnings
warnings.filterwarnings("ignore")

matplotlib.rcParams['font.size'] = 20

In [ ]:
def build_MSM(msm_lagtime,assignments):
    counts = TransitionCountEstimator(lagtime=msm_lagtime, count_mode='sliding').fit_fetch(assignments)
    msm = MaximumLikelihoodMSM().fit_fetch(counts)
    return msm

def Girsanov_MSM(msm_lagtime,assignments,reweighting_factors,stationary_pi=None):
    count_estimator = GirsanovReweightingEstimator(lagtime=msm_lagtime,count_mode='sliding')
    counts = count_estimator.fit(data=assignments,reweighting_factors=reweighting_factors).fetch_model()
    if stationary_pi is not None:
        msm = MaximumLikelihoodMSM(stationary_distribution_constraint=stationary_pi).fit_fetch(counts)
    else:
        msm = MaximumLikelihoodMSM().fit_fetch(counts)
    return msm

# We need to further modify assignments_unbiased and assignments_biased
def reorganize_indices(assignments,n_microstates):
    populated_states = np.unique(assignments)
    empty_states = []
    assignments_new = np.zeros(assignments.shape,dtype=np.int32)
    shift = 0
    for idx in range(n_microstates):
        if idx in populated_states:
            idx_new = idx - shift
            assignments_new[np.where(assignments == idx)] = idx_new
        else:
            empty_states.append(idx)
            shift += 1
    return assignments_new, empty_states

def reorganize_eigenvectors(msm,n_eigenvectors,n_microstates,empty_states,align=None):
    eigenvectors_new = np.zeros((n_microstates,n_eigenvectors))
    shift = 0
    for idx in range(n_microstates):
        if idx in empty_states:
            shift += 1
            empty_entries = np.zeros(n_eigenvectors,)
            # Let values of eigenvectors being nan for plotting
            empty_entries[:] = np.nan
            eigenvectors_new[idx,:] = empty_entries
        else:
            idx_new = idx - shift
            pi_idx = msm.stationary_distribution[idx_new]
            ev_idx = msm.eigenvectors_right()[idx_new,1:n_eigenvectors]
            eigenvectors_new[idx,:] = np.concatenate([[pi_idx],ev_idx])
    if align is not None:
        # Align each eigenvector with the reference eigenvector
        for i in range(1,n_eigenvectors):
            if np.nansum(np.abs(-eigenvectors_new[:,i] - align[:,i])) < np.nansum(np.abs(eigenvectors_new[:,i] - align[:,i])):
                eigenvectors_new[:,i] = -eigenvectors_new[:,i]
    return eigenvectors_new

### 0. Implicit Solvent Ala2 simulation with OpenMM

#### 0.1 System Setup

In [ ]:
# Parameters
pdb_file = 'traj_and_dat/ala2/input.pdb'
protein_ff = 'amber99sbildn.xml'
solvent_ff = 'amber99_obc.xml'      # obc is implicit solvent model
temp = 310
temperature = temp * kelvin
friction = 100 / picosecond
timestep = 2 * femtosecond
splitting = 'R V O V R'
nstxout = 50
kt = temp*8.314/1000
platform = Platform.getPlatformByName('CUDA')

#### 0.2 Unbiased Ref Simulation

In [ ]:
simulation_length = 500000000          # 1us
reporting_frequency = 500000           # 1ns
n_simulations = 1                      # Run 5 trajectories

for i in range(n_simulations):
    pdb = PDBFile(pdb_file)
    forcefield = ForceField(protein_ff,solvent_ff)
    system = forcefield.createSystem(pdb.topology,constraints=HBonds)
    integrator = LangevinSplittingGirsanov(nstxout=nstxout,temperature=temperature,collision_rate=friction, 
                                           timestep=timestep,splitting=splitting)
    
    simulation = Simulation(pdb.topology,system,integrator)
    simulation.context.setPositions(pdb.positions)
    simulation.minimizeEnergy()
    simulation.reporters.append(DCDReporter('traj_and_dat/ala2/ala2-ref.dcd'.format(i=i),nstxout))
    simulation.reporters.append(StateDataReporter(
        stdout,reporting_frequency,step=True,remainingTime=True,
        speed=True,totalSteps=simulation_length))
    simulation.step(simulation_length)

### 1. Phi, Psi as 1D reaction coordinate

#### 1.1 Build-up Only (310K)
Obtain stationary distribution with the assumption of pseudostatic bias.

In [ ]:
# Metadynamics parameters
biasfactor = 1.25
cv_sigma = 0.1
periodic_cv = True
gridwidth = 100
height = 1.2 * kilojoule_per_mole
pace = 500

In [ ]:
# Phi Build-up Simulation
name = 'phi_build_up'
simulation_length = 50000000              # 100ns
n_iters = int(simulation_length / nstxout)       
reporting_frequency = 50000               # 10ps

M = np.zeros(n_iters,)

topology = PDBFile(pdb_file).topology
forcefield = ForceField(protein_ff,solvent_ff)

system = forcefield.createSystem(topology,constraints=HBonds)

# Define the collective variable as the phi angle, C-N-CA-C
# (idx in pdb 5,7,9,15, remember -1 since openmm starts with 0)

cv_force = CustomTorsionForce("theta")
cv_force.addTorsion(4,6,8,14,[])

# Create the BiasVariable and Metadynamics objects
cv = BiasVariable(
    cv_force, -np.pi, np.pi, cv_sigma, periodic_cv, gridwidth
)
    
metad = Metadynamics(
    system, [cv], temperature, biasfactor, height, pace#, save_freq, save_dir
)

# Set ForceGroup to 1 for Girsanov path weights
metad._force.setForceGroup(1)

integrator = LangevinSplittingGirsanov(nstxout=nstxout,temperature=temperature,collision_rate=friction, 
                                       timestep=timestep,splitting=splitting)

simulation = Simulation(pdb.topology,system,integrator)
simulation.context.setPositions(pdb.positions)
simulation.minimizeEnergy()
simulation.reporters.append(DCDReporter('traj_and_dat/ala2/{name}.dcd'.format(name=name),nstxout))
simulation.reporters.append(StateDataReporter(
    stdout,reporting_frequency,step=True,remainingTime=True,
    speed=True,totalSteps=simulation_length))

for k in range(n_iters):
    metad.step(simulation,nstxout)
    M[k] = simulation.integrator.getGlobalVariableByName("M")

np.save('traj_and_dat/ala2/M_{name}.npy'.format(name=name), M)
np.save('traj_and_dat/ala2/Bias_{name}.npy'.format(name=name), metad._totalBias)

In [ ]:
# Psi Build-up Simulation
name = 'psi_build_up'
simulation_length = 25000000              # 50ns
n_iters = int(simulation_length / nstxout)       
reporting_frequency = 50000               # 10ps

M = np.zeros(n_iters,)

topology = PDBFile(pdb_file).topology
forcefield = ForceField(protein_ff,solvent_ff)

system = forcefield.createSystem(topology,constraints=HBonds)

# Define the collective variable as the phi angle, N-CA-C-N
# (idx in pdb 7,9,15,17 remember -1 since openmm starts with 0)

cv_force = CustomTorsionForce("theta")
cv_force.addTorsion(6,8,14,16,[])

# Create the BiasVariable and Metadynamics objects
cv = BiasVariable(
    cv_force, -np.pi, np.pi, cv_sigma, periodic_cv, gridwidth
)
    
metad = Metadynamics(
    system, [cv], temperature, biasfactor, height, pace#, save_freq, save_dir
)

# Set ForceGroup to 1 for Girsanov path weights
metad._force.setForceGroup(1)

integrator = LangevinSplittingGirsanov(nstxout=nstxout,temperature=temperature,collision_rate=friction, 
                                       timestep=timestep,splitting=splitting)

simulation = Simulation(pdb.topology,system,integrator)
simulation.context.setPositions(pdb.positions)
simulation.minimizeEnergy()
simulation.reporters.append(DCDReporter('traj_and_dat/ala2/{name}.dcd'.format(name=name),nstxout))
simulation.reporters.append(StateDataReporter(
    stdout,reporting_frequency,step=True,remainingTime=True,
    speed=True,totalSteps=simulation_length))

for k in range(n_iters):
    metad.step(simulation,nstxout)
    M[k] = simulation.integrator.getGlobalVariableByName("M")

np.save('traj_and_dat/ala2/M_{name}.npy'.format(name=name), M)
np.save('traj_and_dat/ala2/Bias_{name}.npy'.format(name=name), metad._totalBias)